# Competitive Pricing Intelligence & Simulator

**延續 Notebook 1** 的定價模型，**Notebook 2**處理專案真正的核心問題：

> 競爭對手改變價格之後，我們應該怎麼定價，才能讓利潤最大化？

本篇涵蓋：
1. Competitor Price Monitoring
2. 原需求模型反思（無法表現價格戰）
3. Competitive Strategy Simulator
4. Scenario Analysis

**Base Assumptions（沿用 Notebook 1）**：Base Price = 690, Base Demand = 1000, Unit Cost = 400, Elasticity = -1.8


## 1. Competitor Price Monitoring

使用模擬的競品歷史價格資料（`data/competitor_prices.csv`），包含 4 個競品（Anker、Xiaomi、ROMOSS、PhoneMax）60 天的每日價格，並內建 3 次模擬促銷降價事件，用來驗證後續的價格警示機制是否有效。

> ⚠️ 此資料為模擬資料，非真實爬取結果，用於展示分析方法論。


In [1]:
import sys
sys.path.insert(0, "/home/claude/pricing-project/src")


In [2]:
import pandas as pd
from competitor_analysis import load_competitor_data, get_current_prices, build_comparison_table, detect_price_alerts

df = load_competitor_data("/home/claude/pricing-project/data/competitor_prices.csv")
current_prices = get_current_prices(df)

our_price = 690
comparison = build_comparison_table(current_prices, our_price)
comparison.round(1)


,Competitor,Our Price,Competitor Price,Price Diff,Price Diff %
0,Xiaomi,690,496,194,39.1
1,Anker,690,802,-112,-14.0
2,ROMOSS,690,468,222,47.4
3,PhoneMax,690,598,92,15.4


**觀察**：我們（690）比 Xiaomi、ROMOSS、PhoneMax 都貴，但比 Anker 便宜；這代表我們目前在市場上是「中高價位」定位，不是最低價，也不是最高價。


In [3]:
alerts = detect_price_alerts(df, lookback_days=14, threshold=0.05)
alerts.round(1) if not alerts.empty else "目前沒有觸發警示的競品（過去 14 天內）"


/tmp/ipykernel_482/3735719501.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  alerts.round(1) if not alerts.empty else "目前沒有觸發警示的競品（過去 14 天內）"


,Competitor,Latest Price,Price (14 days ago),Change %,Date
0,ROMOSS,468,382,22.5,2026-08-16


## 2. 反思：為什麼原本的需求模型看不到「價格戰」？

在第一版的 Strategy Simulator 裡，我們直接套用 Notebook 1 的 `Q = Q0 * (P/P0)^elasticity`，發現一個違反直覺的現象：**不管競品降到多低，模型幾乎永遠建議「維持原價」**。

**數學原因**：這是一個乘法可分離（multiplicatively separable）的函數形式。就算加入競品價格作為額外的乘法項：

$$Q(P, P_{comp}) = Q_0 \times (P/P_0)^{\varepsilon} \times (P_{comp}/P_{comp,0})^{\varepsilon_{cross}}$$

競品價格項對所有候選策略價格來說是同一個常數，只會等比例縮放所有策略的利潤，**不會改變哪個策略利潤最高的排名**。可以用一階條件證明：這種函數形式下，理論最佳的自身價格，數學上必然獨立於競品價格，是函數形式的結構性限制。


## 3. 解法：改用線性（Bertrand-style）需求模型

要讓「最佳價格」真正隨競品價格移動，改用經濟學處理差異化寡占競爭常用的線性可加模型：

$$Q(P, P_{comp}) = Q_0 + slope_{own} \times (P - P_0) + slope_{cross} \times (P_{comp} - P_{comp,0})$$

其中：
- $slope_{own} = \varepsilon \times Q_0 / P_0$（負值，沿用原本的 elasticity 假設）
- $slope_{cross} = \varepsilon_{cross} \times Q_0 / P_{comp,0}$（正值，新增的 cross-price elasticity 假設）

這是線性可加結構，競品價格的變動會實質改變利潤函數的一階條件，讓最佳自身價格真正對競品價格敏感。


In [4]:
from strategy_simulator import compare_strategies

P0, Q0, elasticity, unit_cost = 690, 1000, -1.8, 400
cross_elasticity = 1.0  # 假設：行動電源接近完全替代品，交叉彈性設中等偏高
competitor_base_price = 802  # Anker 目前價格

print("=== Anker 從 802 降到不同程度，看 Recommended Strategy 怎麼變 ===\n")
for competitor_new_price in [600, 500, 400, 300]:
    strategies = {
        "Maintain (690)": 690,
        f"Match ({competitor_new_price})": competitor_new_price,
        "Partial Cut (600)": 600,
    }
    result_df = compare_strategies(strategies, competitor_new_price, P0, Q0, elasticity,
                                    competitor_base_price, cross_elasticity, unit_cost)
    recommended = result_df.loc[result_df["is_recommended"], "strategy"].values[0]
    print(f"Anker -> {competitor_new_price}: Recommended = {recommended}")
    print(result_df[["strategy", "price", "demand", "profit"]].round(1).to_string(index=False))
    print()


=== Anker 從 802 降到不同程度，看 Recommended Strategy 怎麼變 ===

Anker -> 600: Recommended = Maintain (690)
         strategy  price  demand   profit
   Maintain (690)    690   748.1 216957.6
      Match (600)    600   982.9 196582.5
Partial Cut (600)    600   982.9 196582.5

Anker -> 500: Recommended = Maintain (690)
         strategy  price  demand   profit
   Maintain (690)    690   623.4 180798.0
      Match (500)    500  1119.1 111909.4
Partial Cut (600)    600   858.2 171644.8

Anker -> 400: Recommended = Partial Cut (600)
         strategy  price  demand   profit
   Maintain (690)    690   498.8 144638.4
      Match (400)    400  1255.3      0.0
Partial Cut (600)    600   733.5 146707.1

Anker -> 300: Recommended = Partial Cut (600)
         strategy  price  demand    profit
   Maintain (690)    690   374.1  108478.8
      Match (300)    300  1391.5 -139145.6
Partial Cut (600)    600   608.8  121769.5



**關鍵發現：策略排名真的會隨競品降價幅度而改變。**

| Anker 降到 | Recommended |
|---|---|
| 600 | 維持原價 |
| 500 | 維持原價（跟進的利潤差距已明顯縮小）|
| **400** | **改成部分降價才是最佳解** |
| 300 | 部分降價；完全跟進到底甚至會**虧損** |

**商業意義**：面對競品降價，「該不該反應」與「反應多少」不是固定答案，取決於競品降價幅度是否逼近自身成本線。低於某個底線的價格戰，參與反而不划算。


## 4. Scenario Analysis

系統化比較不同市場條件下，最佳定價 / 最佳策略是否改變。分兩類情境：

- **情境一、成本上升，彈性變敏感**：套用 Notebook 1 的 optimizer，因為這兩者是「我們自己」的參數變化。
- **情境二、競品降價 5%／10%**：套用本篇的競爭型模型，因為只有這個模型能表現出策略排名隨競品價格移動。


In [5]:
from scenario_analysis import run_cost_elasticity_scenarios, run_competitor_price_scenarios

print("情境一：自身成本 / 彈性敏感度分析")
df1 = run_cost_elasticity_scenarios(P0, Q0, elasticity, unit_cost, price_min=400, price_max=1500)
print(df1.round(2).to_string(index=False))


情境一：自身成本 / 彈性敏感度分析
                        scenario  elasticity  unit_cost  optimal_price  expected_profit  profit_margin  price_change_vs_baseline
                        Baseline       -1.80      400.0            900        309928.74           0.56                         0
                  Unit Cost +10%       -1.80      440.0            990        287175.71           0.56                        90
Elasticity More Sensitive (x1.3)       -2.34      400.0            700        290067.26           0.43                      -200


**解讀**：成本上升 10%，最佳價格同比例上調（+90），但利潤率不變（55.6%）。因為利潤率只由彈性決定（`margin = -1/elasticity`），與成本無關，代表成本上升時應該把成本轉嫁出去。彈性變敏感（市場競爭加劇）則同時壓低最佳價格與利潤率，影響比單純成本上升更劇烈。


In [6]:
print("情境二：Anker 降價 5% / 10%")
df2 = run_competitor_price_scenarios(competitor_base_price, P0, Q0, elasticity, unit_cost, cross_elasticity)
df2.round(1)


情境二：Anker 降價 5% / 10%


,scenario,competitor_new_price,recommended_strategy,recommended_profit,maintain_profit,profit_gain_vs_maintain
0,Competitor Price -5%,761.9,Partial Cut,279084.1,275500.0,3584.1
1,Competitor Price -10%,721.8,Match,262924.6,261000.0,1924.6


**解讀**：因為 Anker 原售價（802）比我們（690）貴很多，就算降價 10%，新價格（721.8）仍較高，威脅性有限，模型建議的反應幅度也小（利潤增益 <1.5%）。這與「Anker 降價至 300」的極端情境形成對比，說明模型的建議會隨威脅程度合理縮放。

## 5. 結論與限制

**Key Findings**
- 純乘法（constant elasticity）需求模型結構上無法表現「策略排名隨競品價格改變」，改用線性可加模型後才能捕捉價格戰動態。
- 面對競品降價，是否反應、反應多少，取決於降價幅度是否逼近我們自己的成本線，而非「有降價就要跟」。
- 成本上升與市場競爭加劇（彈性變敏感）對定價策略的影響機制不同：前者可轉嫁、後者直接壓縮利潤空間。

**Limitations**
- Cross-price elasticity 為假設值，且需依競品品牌定位手動調整（不同品牌與我們的替代性不同），未做到自動校準。
- Scenario Analysis 目前僅示範 Anker 一個競品，未系統化涵蓋所有競品組合。
- 所有分析基於模擬競品資料，非真實市場觀測。

**Future Improvements**
- 若能取得真實銷售與競品價格歷史資料，可用迴歸方法估計真實的 own-price 與 cross-price elasticity，取代目前的假設值。
- 可將 cross-price elasticity 依品牌分別設定並儲存，取代目前每次手動調整的方式。
